#Euro 2024 — Goalkeeper Performance Analysis

#StatsBomb Open Data  via statsbombpy
#UEFA Euro 2024  ID:55 Season:282


---

| Section | Metrics |
|---|---|
| 1. Load Data | 51 Euro 2024 matches, all events |
| 2. Shot-Stopping | Shots faced · saves · save % · xG faced |
| 3. xG Prevention | Goals Prevented = xG Against − Goals Conceded |
| 4. GK Actions | Sweeper · claims · punches per match |
| 5. Distribution | Goal kick frequency |
| 6. **Tournament Ranking** | **Composite score across all metrics** |
| 7. Radar Chart | Top 5 GKs visualized |
| 8. Shot Map | Heatmap for #1 ranked GK |

> **Note:** If you cloned `statsbomb/open-data` locally, set `LOCAL_DATA_PATH` in the config cell below.
> Otherwise the library streams it from GitHub automatically.


In [1]:
# Install StatsBomb Python library (free open data — no credentials needed)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "statsbombpy", "-q"])
print("statsbombpy ready!")


statsbombpy ready!


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from statsbombpy import sb
from mplsoccer import VerticalPitch
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── Colour palette ──
BG     = '#0d1117'
PITCH  = '#161b22'
WHITE  = '#e6edf3'
BLUE   = '#58a6ff'
GREEN  = '#3fb950'
RED    = '#f85149'
YELLOW = '#d29922'

plt.rcParams.update({
    'figure.facecolor': BG,
    'axes.facecolor':   PITCH,
    'text.color':       WHITE,
    'axes.labelcolor':  WHITE,
    'xtick.color':      WHITE,
    'ytick.color':      WHITE,
    'axes.edgecolor':   '#30363d',
    'grid.color':       '#21262d',
    'axes.titlecolor':  WHITE,
})
print("Libraries loaded ✓")


Libraries loaded ✓


In [8]:
# ── CONFIG ─────────────────────────────────────────────────────────────────────
# If you have the statsbomb/open-data repo cloned locally, set the path here.
# e.g. LOCAL_DATA_PATH = r"C:/Users/prash/open-data"
# Leave as None to stream directly from GitHub (statsbombpy default).

LOCAL_DATA_PATH = None   # ← change this if you have local data

COMPETITION_ID = 55    # UEFA Euro
SEASON_ID      = 282   # 2024
MIN_MATCHES    = 2     # GKs with fewer matches are excluded from ranking


## 1 · Load Euro 2024 Data

In [4]:
if LOCAL_DATA_PATH:
    # Point statsbombpy to local clone
    import os
    os.environ['STATSBOMB_DATA_ROOT'] = LOCAL_DATA_PATH

matches = sb.matches(competition_id=COMPETITION_ID, season_id=SEASON_ID)
print(f"Euro 2024: {len(matches)} matches loaded")

cols = ['match_id', 'home_team', 'away_team', 'home_score', 'away_score']
if 'competition_stage' in matches.columns:
    cols = ['competition_stage'] + cols
matches[cols].head(10)


Euro 2024: 51 matches loaded


,competition_stage,match_id,home_team,away_team,home_score,away_score
0,Group Stage,3930166,Portugal,Czech Republic,2,1
1,Group Stage,3930159,Hungary,Switzerland,1,3
2,Group Stage,3930163,Serbia,England,0,1
3,Group Stage,3930181,England,Slovenia,0,0
4,Group Stage,3930178,Croatia,Italy,1,1
5,Group Stage,3930160,Spain,Croatia,3,0
6,Group Stage,3930170,Slovenia,Serbia,1,1
7,Group Stage,3938643,France,Poland,1,1
8,Group Stage,3938641,Poland,Austria,1,3
9,Group Stage,3938642,Georgia,Czech Republic,1,1


In [5]:
print("Loading events for all matches (≈ 1-2 min)...")

all_events, failed = [], []
for i, mid in enumerate(matches['match_id']):
    try:
        ev = sb.events(match_id=mid)
        ev['match_id'] = mid
        all_events.append(ev)
    except Exception as e:
        failed.append((mid, str(e)))
    if (i + 1) % 10 == 0 or (i + 1) == len(matches):
        print(f"  {i+1}/{len(matches)} loaded...")

events = pd.concat(all_events, ignore_index=True)
if failed:
    print(f"Failed matches: {[f[0] for f in failed]}")
print(f"\nTotal events: {len(events):,}")
print(f"Columns sample: {list(events.columns[:12])}")


Loading events for all matches (≈ 1-2 min)...
  10/51 loaded...
  20/51 loaded...
  30/51 loaded...
  40/51 loaded...
  50/51 loaded...
  51/51 loaded...

Total events: 187,924
Columns sample: ['50_50', 'bad_behaviour_card', 'ball_receipt_outcome', 'ball_recovery_recovery_failure', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_other', 'clearance_right_foot', 'counterpress']


In [6]:
# statsbombpy may return nested dicts or already-flattened columns
# depending on version. This cell handles both.

def safe_get(val, *keys, default=''):
    for k in keys:
        if isinstance(val, dict):
            val = val.get(k, default)
        else:
            return default
    return val

def flatten(series, *keys):
    return series.apply(lambda x: safe_get(x, *keys))

# Detect format
sample = events['type'].iloc[0] if 'type' in events.columns else None
IS_NESTED = isinstance(sample, dict)
print(f"Data format: {'nested dicts' if IS_NESTED else 'already flat'}")

if IS_NESTED:
    events['type_name']     = flatten(events['type'], 'name')
    events['player_name']   = flatten(events['player'], 'name')
    events['player_id']     = flatten(events['player'], 'id')
    events['team_name']     = flatten(events['team'],   'name')
    pos_col = events['position'] if 'position' in events.columns else pd.Series([{}]*len(events))
    events['position_name'] = flatten(pos_col, 'name')
else:
    # Already flat — check column names
    rename = {'type':'type_name', 'player':'player_name', 'team':'team_name', 'position':'position_name'}
    for old, new in rename.items():
        if old in events.columns and new not in events.columns:
            events[new] = events[old]

print("Top event types:")
print(events['type_name'].value_counts().head(10).to_string())


Data format: already flat
Top event types:
type_name
Pass             53888
Ball Receipt*    51608
Carry            44157
Pressure         14534
Ball Recovery     4144
Duel              3069
Block             2002
Clearance         1834
Goal Keeper       1622
Shot              1340


## 2 · Extract Goalkeeper Data

In [ ]:
shots_raw = events[events['type_name'] == 'Shot'].copy()
print(f"Total shots in tournament: {len(shots_raw)}")

if IS_NESTED:
    shots_raw['xg']          = shots_raw['shot'].apply(lambda x: safe_get(x, 'statsbomb_xg', default=0) if isinstance(x, dict) else 0)
    shots_raw['outcome']     = shots_raw['shot'].apply(lambda x: safe_get(x, 'outcome', 'name'))
    shots_raw['gk_name']     = shots_raw['shot'].apply(lambda x: safe_get(x, 'goalkeeper', 'name'))
    shots_raw['body_part']   = shots_raw['shot'].apply(lambda x: safe_get(x, 'body_part', 'name'))
    shots_raw['technique']   = shots_raw['shot'].apply(lambda x: safe_get(x, 'technique', 'name'))
    shots_raw['shot_x']      = shots_raw['location'].apply(lambda x: x[0] if isinstance(x, list) and len(x)>0 else None)
    shots_raw['shot_y']      = shots_raw['location'].apply(lambda x: x[1] if isinstance(x, list) and len(x)>1 else None)
else:
    # Flat format — derive GK name from the opposing team's goalkeeper per match
    xg_col = next((c for c in shots_raw.columns if 'statsbomb_xg' in c), None)
    outcome_col = next((c for c in shots_raw.columns if c in ('shot_outcome', 'shot_outcome_name')), None)
    if xg_col:
        shots_raw['xg'] = shots_raw[xg_col]
    if outcome_col:
        shots_raw['outcome'] = shots_raw[outcome_col]
    if 'location' in shots_raw.columns:
        shots_raw['shot_x'] = shots_raw['location'].apply(lambda x: x[0] if isinstance(x, list) and len(x)>0 else None)
        shots_raw['shot_y'] = shots_raw['location'].apply(lambda x: x[1] if isinstance(x, list) and len(x)>1 else None)

    # Build match → team → GK map from position data
    gk_rows = events[events['position_name'].str.contains('Goalkeeper', case=False, na=False)][
        ['match_id', 'team_name', 'player_name']].drop_duplicates()
    gk_map = {(r['match_id'], r['team_name']): r['player_name'] for _, r in gk_rows.iterrows()}
    match_teams = events.groupby('match_id')['team_name'].unique()

    def find_opposing_gk(row):
        for t in match_teams.get(row['match_id'], []):
            if t != row['team_name']:
                return gk_map.get((row['match_id'], t), '')
        return ''

    shots_raw['gk_name'] = shots_raw.apply(find_opposing_gk, axis=1)

shots_raw['xg'] = pd.to_numeric(shots_raw.get('xg', 0), errors='coerce').fillna(0)
shots = shots_raw[shots_raw['gk_name'].notna() & (shots_raw['gk_name'] != '')].copy()

print(f"Shots with GK info: {len(shots)}")
print(f"Outcomes: {shots['outcome'].value_counts().to_dict()}")
shots[['player_name','team_name','gk_name','xg','outcome','shot_x','shot_y']].head()

In [ ]:
gk_ev = events[events['type_name'] == 'Goal Keeper'].copy()
print(f"Total GK events: {len(gk_ev)}")

if IS_NESTED:
    gk_ev['gk_action']  = gk_ev['goalkeeper'].apply(lambda x: safe_get(x, 'type', 'name') if isinstance(x, dict) else '')
    gk_ev['gk_outcome'] = gk_ev['goalkeeper'].apply(lambda x: safe_get(x, 'outcome', 'name') if isinstance(x, dict) else '')
else:
    gk_ev['gk_action']  = gk_ev.get('goalkeeper_type_name', pd.Series([''] * len(gk_ev)))
    gk_ev['gk_outcome'] = gk_ev.get('goalkeeper_outcome_name', pd.Series([''] * len(gk_ev)))

print("\nGK action types:")
print(gk_ev['gk_action'].value_counts().to_string())


In [ ]:
passes_all = events[events['type_name'] == 'Pass'].copy()

if IS_NESTED:
    passes_all['pass_type']   = passes_all['pass'].apply(lambda x: safe_get(x, 'type', 'name') if isinstance(x, dict) else '')
    passes_all['pass_length'] = passes_all['pass'].apply(lambda x: safe_get(x, 'length', default=0) if isinstance(x, dict) else 0)
    passes_all['pass_height'] = passes_all['pass'].apply(lambda x: safe_get(x, 'height', 'name') if isinstance(x, dict) else '')
else:
    passes_all['pass_type']   = passes_all.get('pass_type_name', '')
    passes_all['pass_length'] = pd.to_numeric(passes_all.get('pass_length', 0), errors='coerce').fillna(0)

goal_kicks = passes_all[passes_all['pass_type'] == 'Goal Kick'].copy()
print(f"Goal kicks in tournament: {len(goal_kicks)}")
print(f"Average length: {pd.to_numeric(goal_kicks['pass_length'], errors='coerce').mean():.1f} m")


In [ ]:
# ── Shot-stopping stats ──────────────────────────────────────────────────────
gk_shots = shots.groupby('gk_name').agg(
    shots_faced      = ('xg',      'count'),
    xg_faced         = ('xg',      'sum'),
    goals_conceded   = ('outcome', lambda x: (x == 'Goal').sum()),
    shots_on_target  = ('outcome', lambda x: x.isin(['Saved','Goal']).sum()),
    saves            = ('outcome', lambda x: (x == 'Saved').sum()),
    blocked_shots    = ('outcome', lambda x: (x == 'Blocked').sum()),
).reset_index()

gk_shots['save_pct']        = (gk_shots['saves'] / gk_shots['shots_on_target'].replace(0, np.nan) * 100).fillna(0)
gk_shots['goals_prevented'] = gk_shots['xg_faced'] - gk_shots['goals_conceded']

# ── GK action counts ─────────────────────────────────────────────────────────
gk_actions = gk_ev.groupby('player_name').agg(
    sweeper_actions = ('gk_action', lambda x: (x == 'Keeper Sweeper').sum()),
    claims          = ('gk_action', lambda x: (x == 'Collected').sum()),   # StatsBomb: 'Collected' not 'Claim'
    punches         = ('gk_action', lambda x: (x == 'Punch').sum()),
    total_gk_events = ('gk_action', 'count'),
).reset_index().rename(columns={'player_name': 'gk_name'})

# ── Goal kick counts ──────────────────────────────────────────────────────────
gk_gks = goal_kicks.groupby('player_name').agg(
    goal_kicks       = ('pass_type',   'count'),
    avg_gk_length    = ('pass_length', 'mean'),
).reset_index().rename(columns={'player_name': 'gk_name'})

# ── Matches played ────────────────────────────────────────────────────────────
gk_mp = gk_ev.groupby('player_name')['match_id'].nunique().reset_index()
gk_mp.columns = ['gk_name', 'matches_played']

# ── Merge ─────────────────────────────────────────────────────────────────────
gk = (gk_shots
      .merge(gk_actions, on='gk_name', how='left')
      .merge(gk_gks,     on='gk_name', how='left')
      .merge(gk_mp,      on='gk_name', how='left')
      .fillna(0))

mp = gk['matches_played'].replace(0, 1)
gk['sweeper_per_match']   = gk['sweeper_actions'] / mp
gk['claims_per_match']    = gk['claims'] / mp
gk['punches_per_match']   = gk['punches'] / mp
gk['goal_kicks_per_match']= gk['goal_kicks'] / mp
gk['gp_per_match']        = gk['goals_prevented'] / mp
gk['xg_per_match']        = gk['xg_faced'] / mp

# Filter minimum matches
gk = gk[gk['matches_played'] >= MIN_MATCHES].copy()
print(f"GKs with ≥{MIN_MATCHES} matches: {len(gk)}")

gk.sort_values('goals_prevented', ascending=False)[
    ['gk_name','matches_played','shots_faced','saves','save_pct','xg_faced','goals_conceded','goals_prevented']
]

## 3 · Shot-Stopping: Save Percentage

In [ ]:
top = gk.sort_values('save_pct', ascending=True)
med = gk['save_pct'].median()

fig, ax = plt.subplots(figsize=(12, 7))
colors = [GREEN if v >= med else RED for v in top['save_pct']]
bars = ax.barh(top['gk_name'], top['save_pct'], color=colors, edgecolor='none', height=0.65)

for bar, val in zip(bars, top['save_pct']):
    ax.text(val + 0.4, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', ha='left', fontsize=9.5, fontweight='bold', color=WHITE)

ax.axvline(med, color=YELLOW, ls='--', lw=1.5, alpha=0.8, label=f'Median: {med:.1f}%')
ax.set_xlabel('Save Percentage  (%)', fontsize=11)
ax.set_title('Euro 2024 — Goalkeeper Save Percentage', fontsize=14, fontweight='bold', pad=14)
ax.set_xlim(0, 110)
ax.legend(framealpha=0.25)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('gk_save_pct.png', dpi=150, bbox_inches='tight')
plt.show()


## 4 · xG Prevention Scatter

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

sc = ax.scatter(
    gk['xg_faced'], gk['goals_conceded'],
    c=gk['goals_prevented'], cmap='RdYlGn',
    s=gk['matches_played'] * 35 + 40,
    edgecolors='white', linewidths=0.6, alpha=0.92,
    vmin=gk['goals_prevented'].min(), vmax=gk['goals_prevented'].max()
)

# Expected line: xG = GA
lim = max(gk['xg_faced'].max(), gk['goals_conceded'].max()) + 0.5
ax.plot([0, lim], [0, lim], color='#8b949e', ls='--', lw=1.3, alpha=0.6, label='Expected (xG = GA)')

for _, row in gk.iterrows():
    ax.annotate(row['gk_name'].split()[-1],
                (row['xg_faced'], row['goals_conceded']),
                xytext=(5, 4), textcoords='offset points',
                fontsize=7.5, color=WHITE, alpha=0.88)

cbar = plt.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label('Goals Prevented  (xG − GA)', color=WHITE, fontsize=10)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=WHITE)

ax.set_xlabel('Total xG Faced', fontsize=11)
ax.set_ylabel('Goals Conceded', fontsize=11)
ax.set_title('Euro 2024 GKs — xG Faced vs Goals Conceded\n'
             '(Green = more goals prevented · bubble size = matches played)', fontsize=13, fontweight='bold', pad=14)
ax.text(0.02, 0.97, '▲ Below diagonal = fewer goals than xG predicted (elite)',
        transform=ax.transAxes, fontsize=8.5, color='#56d364', va='top')
ax.legend(framealpha=0.25)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('gk_xg_prevention.png', dpi=150, bbox_inches='tight')
plt.show()


## 5 · GK Actions per Match

In [ ]:
top10 = gk.nlargest(min(10, len(gk)), 'total_gk_events').copy()
x = np.arange(len(top10))
width = 0.6

fig, ax = plt.subplots(figsize=(13, 6))
bottom = np.zeros(len(top10))

for col, color, label in [
    ('sweeper_per_match', BLUE,   'Sweeper Actions / match'),
    ('claims_per_match',  GREEN,  'Claims / match'),
    ('punches_per_match', YELLOW, 'Punches / match'),
]:
    vals = top10[col].values
    ax.bar(x, vals, width, bottom=bottom, color=color, edgecolor='none', label=label)
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels([n.split()[-1] for n in top10['gk_name']], rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Actions per Match', fontsize=11)
ax.set_title('Euro 2024 — Goalkeeper Actions per Match (Top 10 by Volume)\n'
             'Sweeper Actions · Claims · Punches', fontsize=13, fontweight='bold', pad=14)
ax.legend(framealpha=0.25)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('gk_actions.png', dpi=150, bbox_inches='tight')
plt.show()


## 6 · Goal Kick Distribution

In [ ]:
gk_gk_top = gk.nlargest(min(12, len(gk)), 'goal_kicks_per_match')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar: Goal kicks per match
axes[0].barh(gk_gk_top['gk_name'][::-1], gk_gk_top['goal_kicks_per_match'][::-1],
             color=BLUE, edgecolor='none', height=0.65)
axes[0].set_xlabel('Goal Kicks per Match', fontsize=11)
axes[0].set_title('Goal Kicks per Match', fontsize=12, fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)

# Scatter: avg GK length vs GP per match
sc = axes[1].scatter(
    gk['avg_gk_length'], gk['gp_per_match'],
    c=gk['save_pct'], cmap='YlGn', s=80, edgecolors='white', lw=0.5
)
for _, row in gk.iterrows():
    axes[1].annotate(row['gk_name'].split()[-1],
                     (row['avg_gk_length'], row['gp_per_match']),
                     xytext=(3, 3), textcoords='offset points', fontsize=7, color=WHITE, alpha=0.8)
cbar2 = plt.colorbar(sc, ax=axes[1])
cbar2.set_label('Save %', color=WHITE)
plt.setp(cbar2.ax.yaxis.get_ticklabels(), color=WHITE)
axes[1].set_xlabel('Avg Goal Kick Length (m)', fontsize=11)
axes[1].set_ylabel('Goals Prevented per Match', fontsize=11)
axes[1].set_title('GK Length vs Goals Prevented/match', fontsize=12, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Euro 2024 — Goal Kick Distribution Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('gk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 7 · Full Tournament Ranking

In [ ]:
def norm(series, ascending=True):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    n = (series - mn) / (mx - mn)
    return n if ascending else 1 - n

gk_r = gk.copy()
gk_r['n_save_pct']   = norm(gk_r['save_pct'])
gk_r['n_gp']         = norm(gk_r['goals_prevented'])        # xG - GA
gk_r['n_sweeper']    = norm(gk_r['sweeper_per_match'])
gk_r['n_claims']     = norm(gk_r['claims_per_match'])
gk_r['n_ga']         = norm(gk_r['goals_conceded'], ascending=False)  # fewer = better

# Weights: GP 35% | Save% 30% | Sweeper 15% | Claims 10% | GA 10%
weights = {'n_save_pct': 0.30, 'n_gp': 0.35, 'n_sweeper': 0.15, 'n_claims': 0.10, 'n_ga': 0.10}
gk_r['score'] = sum(gk_r[c] * w for c, w in weights.items())
gk_r = gk_r.sort_values('score', ascending=False).reset_index(drop=True)
gk_r.index += 1

ranking_cols = ['gk_name','matches_played','shots_faced','saves','save_pct',
                'xg_faced','goals_conceded','goals_prevented',
                'sweeper_per_match','claims_per_match','score']
ranking = gk_r[ranking_cols].copy()
ranking.columns = ['Goalkeeper','MP','Shots','Saves','Save %',
                   'xG Faced','GA','Goals Prev.',
                   'Sweeper/M','Claims/M','Score']
print("Euro 2024 Goalkeeper Tournament Ranking")
print("=" * 70)
print(f"Weights: Goals Prevented 35% · Save% 30% · Sweeper 15% · Claims 10% · GA 10%")
print()
ranking


In [ ]:
top_n  = min(15, len(gk_r))
top    = gk_r.head(top_n)
pal    = cm.Blues(np.linspace(0.4, 0.9, top_n))[::-1]

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top['gk_name'][::-1], top['score'][::-1], color=pal, edgecolor='none', height=0.65)

for bar, row in zip(bars, top.iloc[::-1].itertuples()):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
            f'{row.score:.3f}  |  Prev {row.goals_prevented:+.1f}  |  Sv% {row.save_pct:.0f}%',
            va='center', ha='left', fontsize=8.5, color=WHITE)

ax.set_xlabel('Composite Performance Score', fontsize=11)
ax.set_title(f'Euro 2024 — Goalkeeper Rankings  (Top {top_n})\n'
             'Score = Goals Prev. 35% · Save% 30% · Sweeper 15% · Claims 10% · GA 10%',
             fontsize=13, fontweight='bold', pad=14)
ax.set_xlim(0, top['score'].max() * 1.4)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('gk_ranking.png', dpi=150, bbox_inches='tight')
plt.show()


## 8 · Radar Chart — Top 5 Goalkeepers

In [ ]:
top5 = gk_r.head(5)
metrics = ['n_save_pct','n_gp','n_sweeper','n_claims','n_ga']
labels  = ['Save %', 'Goals\nPrevented', 'Sweeper\nActions', 'Claims', 'GA\n(inv.)']

N = len(metrics)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
ax.set_facecolor('#0d1117')
fig.patch.set_facecolor(BG)

# Grid rings
for r in [0.2, 0.4, 0.6, 0.8, 1.0]:
    ax.plot(angles, [r]*(N+1), color='#30363d', lw=0.7, zorder=0)

cmap = plt.cm.tab10
for i, (_, row) in enumerate(top5.iterrows()):
    vals = [row[m] for m in metrics] + [row[metrics[0]]]
    color = cmap(i)
    ax.plot(angles, vals, 'o-', lw=2.2, ms=5, color=color, label=row['gk_name'])
    ax.fill(angles, vals, alpha=0.18, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, color=WHITE, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticklabels([])
ax.spines['polar'].set_color('#30363d')
ax.grid(False)
ax.set_title('Euro 2024 — Top 5 Goalkeepers\n(all metrics normalised 0–1)',
             fontsize=14, fontweight='bold', pad=35, color=WHITE)
ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.12),
          labelcolor=WHITE, facecolor='#161b22', edgecolor='#30363d', fontsize=10)
plt.tight_layout()
plt.savefig('gk_radar.png', dpi=150, bbox_inches='tight')
plt.show()


## 9 · Shot Map — #1 Ranked Goalkeeper

In [ ]:
top_gk = gk_r.iloc[0]['gk_name']
gk_shots_map = shots[shots['gk_name'] == top_gk].copy()

pitch = VerticalPitch(pitch_type='statsbomb', pitch_color=PITCH,
                      line_color='#c7d5cc', half=True, goal_type='box')
fig, ax = pitch.draw(figsize=(8, 10))
fig.patch.set_facecolor(BG)

outcome_style = {
    'Saved':   (GREEN,  'o'),
    'Goal':    (RED,    'X'),
    'Off T':   (BLUE,   's'),
    'Blocked': (YELLOW, 'D'),
    'Post':    ('#ff9500', '^'),
    'Wayward': ('#8b949e', 'P'),
}
for outcome, (color, marker) in outcome_style.items():
    mask = gk_shots_map['outcome'] == outcome
    if mask.sum() == 0:
        continue
    pitch.scatter(
        gk_shots_map.loc[mask, 'shot_x'],
        gk_shots_map.loc[mask, 'shot_y'],
        s=gk_shots_map.loc[mask, 'xg'] * 600 + 40,
        c=color, marker=marker, alpha=0.88,
        edgecolors='white', linewidths=0.5,
        label=f'{outcome} ({mask.sum()})', ax=ax
    )

total, ga, xg_tot = len(gk_shots_map), (gk_shots_map['outcome']=='Goal').sum(), gk_shots_map['xg'].sum()
prev = xg_tot - ga

fig.text(0.5, 0.98, f'Shot Map — {top_gk}', ha='center', fontsize=15, fontweight='bold', color=WHITE)
fig.text(0.5, 0.95,
         f'Shots: {total}  ·  GA: {ga}  ·  xG: {xg_tot:.1f}  ·  Goals Prevented: {prev:+.1f}',
         ha='center', fontsize=10, color='#8b949e')
fig.text(0.5, 0.02, 'Bubble size ∝ xG value', ha='center', fontsize=8, color='#8b949e')
ax.legend(loc='upper left', framealpha=0.28, labelcolor=WHITE,
          facecolor='#161b22', edgecolor='#30363d', fontsize=9)
plt.savefig(f'shot_map_{top_gk.replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
plt.show()


## 10 · Key Insights

In [ ]:
print("=" * 65)
print("EURO 2024 GOALKEEPER PERFORMANCE SUMMARY")
print("=" * 65)

best_save_pct = gk.loc[gk['save_pct'].idxmax()]
best_preventer = gk.loc[gk['goals_prevented'].idxmax()]
best_sweeper  = gk.loc[gk['sweeper_per_match'].idxmax()]
most_busy     = gk.loc[gk['shots_faced'].idxmax()]

print(f"\n🥅 Best Save %:        {best_save_pct['gk_name']:25s}  {best_save_pct['save_pct']:.1f}%")
print(f"🛡️  Most Goals Prevented: {best_preventer['gk_name']:25s}  {best_preventer['goals_prevented']:+.2f}")
print(f"⚡ Best Sweeper/match: {best_sweeper['gk_name']:25s}  {best_sweeper['sweeper_per_match']:.2f}")
print(f"🔥 Busiest GK:         {most_busy['gk_name']:25s}  {int(most_busy['shots_faced'])} shots faced")
print()
print(f"Tournament avg save %:  {gk['save_pct'].mean():.1f}%")
print(f"Tournament avg xG/match:{gk['xg_per_match'].mean():.2f}")
print()
print("TOP 5 RANKED GOALKEEPERS:")
print("-" * 45)
for i, (_, row) in enumerate(gk_r.head(5).iterrows(), 1):
    print(f"  {i}. {row['gk_name']:<25s}  Score: {row['score']:.3f}  |  MP:{int(row['matches_played'])}  |  Prev:{row['goals_prevented']:+.1f}")


## 11 · Green Pitch Save Chart — Top Goalkeeper

In [ ]:
from mplsoccer import VerticalPitch
import matplotlib.patheffects as pe

# ── Pick goalkeeper (change name below to view any GK) ──────────────────────
top_gk = gk_r.iloc[0]['gk_name']   # #1 ranked; replace with e.g. "Manuel Neuer"
gk_map = shots[shots['gk_name'] == top_gk].copy()

# StatsBomb coords: shot origin is from the shooter, so x/y are shooter positions.
# For a GK save map we flip to show from GK's goal perspective (half-pitch).
# StatsBomb pitch: x 0→120, y 0→80. Half=True shows x 60→120 (attacking half).

pitch = VerticalPitch(
    pitch_type='statsbomb',
    pitch_color='grass',        # traditional green with lighter stripes
    line_color='white',
    half=True,
    goal_type='box',
    linewidth=1.5,
    spot_scale=0.01,
)

fig, ax = pitch.draw(figsize=(9, 11))
fig.set_facecolor('#1a472a')    # dark green border to match pitch feel

# ── Plot each shot outcome ───────────────────────────────────────────────────
styles = {
    'Saved':   dict(color='#00e676', marker='o',   label='Saved',         zorder=5),
    'Goal':    dict(color='#ff1744', marker='*',   label='Goal Conceded', zorder=6),
    'Blocked': dict(color='#ffea00', marker='s',   label='Blocked',       zorder=4),
    'Off T':   dict(color='#90caf9', marker='^',   label='Off Target',    zorder=3),
    'Post':    dict(color='#ff9100', marker='D',   label='Post/Bar',      zorder=4),
    'Wayward': dict(color='#b0bec5', marker='v',   label='Wayward',       zorder=3),
}

for outcome, style in styles.items():
    mask = gk_map['outcome'] == outcome
    if mask.sum() == 0:
        continue
    sizes = gk_map.loc[mask, 'xg'] * 800 + 60
    pitch.scatter(
        gk_map.loc[mask, 'shot_x'],
        gk_map.loc[mask, 'shot_y'],
        s=sizes,
        c=style['color'],
        marker=style['marker'],
        alpha=0.88,
        edgecolors='white',
        linewidths=0.6,
        label=f"{style['label']} ({mask.sum()})",
        ax=ax,
        zorder=style['zorder'],
    )

# ── xG labels for high-danger shots (xg > 0.30) ─────────────────────────────
for _, row in gk_map[gk_map['xg'] > 0.30].iterrows():
    ax.annotate(
        f"xG {row['xg']:.2f}",
        (row['shot_y'], row['shot_x']),        # mplsoccer VerticalPitch swaps x/y for annotation
        fontsize=7, color='white', fontweight='bold',
        xytext=(0, 8), textcoords='offset points', ha='center',
        path_effects=[pe.withStroke(linewidth=2, foreground='black')],
    )

# ── Stats summary box ────────────────────────────────────────────────────────
total  = len(gk_map)
ga     = (gk_map['outcome'] == 'Goal').sum()
saves  = (gk_map['outcome'] == 'Saved').sum()
xg_tot = gk_map['xg'].sum()
sv_pct = saves / max((saves + ga), 1) * 100
prev   = xg_tot - ga

stats_text = (
    f"Shots faced: {total}   |   Saves: {saves}   |   Goals: {ga}\n"
    f"Save %: {sv_pct:.1f}%   |   xG faced: {xg_tot:.2f}   |   Goals prevented: {prev:+.2f}"
)

fig.text(0.5, 0.97, top_gk, ha='center', va='top',
         fontsize=17, fontweight='bold', color='white',
         path_effects=[pe.withStroke(linewidth=3, foreground='#1a472a')])
fig.text(0.5, 0.935, 'UEFA Euro 2024 — Shot Map (GK perspective)', ha='center', va='top',
         fontsize=10, color='#c8e6c9',
         path_effects=[pe.withStroke(linewidth=2, foreground='#1a472a')])
fig.text(0.5, 0.905, stats_text, ha='center', va='top',
         fontsize=9, color='white', linespacing=1.6,
         path_effects=[pe.withStroke(linewidth=2, foreground='#1a472a')])
fig.text(0.5, 0.02, 'Bubble size ∝ xG  ·  StatsBomb Open Data',
         ha='center', fontsize=8, color='#a5d6a7',
         path_effects=[pe.withStroke(linewidth=1.5, foreground='#1a472a')])

# ── Legend ────────────────────────────────────────────────────────────────────
legend = ax.legend(
    loc='upper left',
    fontsize=9,
    framealpha=0.6,
    facecolor='#1b5e20',
    edgecolor='white',
    labelcolor='white',
    markerscale=1.2,
)

plt.tight_layout(rect=[0, 0.03, 1, 0.90])
plt.savefig(f'green_save_map_{top_gk.replace(" ", "_")}.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: green_save_map_{top_gk.replace(' ', '_')}.png")


## 12 · Long Ball King — GK Long Pass Distribution

In [ ]:
# ── Identify GK players from their position ──────────────────────────────────
gk_players = events[events['position_name'].str.contains('Goalkeeper', case=False, na=False)][
    ['player_name','player_id']
].drop_duplicates()
gk_player_names = set(gk_players['player_name'].dropna().unique())

# ── All passes made BY goalkeepers ───────────────────────────────────────────
gk_passes = passes_all[passes_all['player_name'].isin(gk_player_names)].copy()
gk_passes['pass_length'] = pd.to_numeric(gk_passes['pass_length'], errors='coerce').fillna(0)

# Extract pass start & end locations
if IS_NESTED:
    gk_passes['pass_start_x'] = gk_passes['location'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
    gk_passes['pass_start_y'] = gk_passes['location'].apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None)
    gk_passes['pass_end_x']   = gk_passes['pass'].apply(lambda x: safe_get(x, 'end_location', default=[None,None])[0] if isinstance(x, dict) else None)
    gk_passes['pass_end_y']   = gk_passes['pass'].apply(lambda x: safe_get(x, 'end_location', default=[None,None])[1] if isinstance(x, dict) else None)
    gk_passes['pass_outcome'] = gk_passes['pass'].apply(lambda x: safe_get(x, 'outcome', 'name') if isinstance(x, dict) else '')
else:
    for col in ['pass_start_x','pass_start_y','pass_end_x','pass_end_y']:
        if col not in gk_passes.columns:
            gk_passes[col] = None
    if 'pass_outcome_name' in gk_passes.columns:
        gk_passes['pass_outcome'] = gk_passes['pass_outcome_name']
    else:
        gk_passes['pass_outcome'] = ''

# Classify: Long = 32m+, Short = <32m (UEFA standard threshold)
LONG_THRESHOLD = 32
gk_passes['is_long']  = gk_passes['pass_length'] >= LONG_THRESHOLD
gk_passes['is_short'] = gk_passes['pass_length'] < LONG_THRESHOLD

print(f"Total GK passes: {len(gk_passes)}")
print(f"Long passes (≥{LONG_THRESHOLD}m): {gk_passes['is_long'].sum()}")
print(f"Short passes (<{LONG_THRESHOLD}m): {gk_passes['is_short'].sum()}")

In [ ]:
# ── Long Ball King Chart ──────────────────────────────────────────────────────
long_passes = gk_passes[gk_passes['is_long']].copy()
long_passes['is_successful'] = long_passes['pass_outcome'].isin(['', 'Complete'])

long_stats = long_passes.groupby('player_name').agg(
    long_balls       = ('is_long',       'sum'),
    successful_long  = ('is_successful', 'sum'),
    avg_long_length  = ('pass_length',   'mean'),
).reset_index()
long_stats = long_stats.merge(gk_mp, left_on='player_name', right_on='gk_name', how='left').drop(columns='gk_name')
long_stats = long_stats[long_stats['matches_played'] >= MIN_MATCHES].copy()
long_stats['long_per_match'] = long_stats['long_balls'] / long_stats['matches_played'].replace(0, 1)
long_stats['long_accuracy']  = (long_stats['successful_long'] / long_stats['long_balls'].replace(0, 1) * 100)
long_stats = long_stats.sort_values('long_per_match', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: Long Balls per Match (horizontal bar)
colors_long = [BLUE if v >= long_stats['long_per_match'].median() else '#4a6fa5' for v in long_stats['long_per_match']]
bars = axes[0].barh(long_stats['player_name'], long_stats['long_per_match'],
                     color=colors_long, edgecolor='none', height=0.65)
for bar, val in zip(bars, long_stats['long_per_match']):
    axes[0].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', ha='left', fontsize=9, fontweight='bold', color=WHITE)
axes[0].set_xlabel('Long Passes per Match (≥32m)', fontsize=11)
axes[0].set_title('Long Ball Frequency', fontsize=13, fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)
med_long = long_stats['long_per_match'].median()
axes[0].axvline(med_long, color=YELLOW, ls='--', lw=1.3, alpha=0.8, label=f'Median: {med_long:.1f}')
axes[0].legend(framealpha=0.25, fontsize=9)

# Right: Accuracy vs Volume scatter
sc = axes[1].scatter(
    long_stats['long_per_match'], long_stats['long_accuracy'],
    c=long_stats['avg_long_length'], cmap='coolwarm',
    s=long_stats['long_balls'] * 3 + 50,
    edgecolors='white', linewidths=0.6, alpha=0.9
)
for _, row in long_stats.iterrows():
    axes[1].annotate(row['player_name'].split()[-1],
                     (row['long_per_match'], row['long_accuracy']),
                     xytext=(5, 4), textcoords='offset points',
                     fontsize=8, color=WHITE, alpha=0.9)
cbar = plt.colorbar(sc, ax=axes[1])
cbar.set_label('Avg Long Pass Length (m)', color=WHITE, fontsize=10)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=WHITE)
axes[1].set_xlabel('Long Passes per Match', fontsize=11)
axes[1].set_ylabel('Long Pass Accuracy (%)', fontsize=11)
axes[1].set_title('Accuracy vs Volume', fontsize=13, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

# Crown the king
king = long_stats.iloc[-1]
plt.suptitle(f'Euro 2024 — Long Ball King: {king["player_name"]}\n'
             f'{king["long_per_match"]:.1f} long balls/match  ·  {king["long_accuracy"]:.0f}% accuracy  ·  Avg {king["avg_long_length"]:.0f}m',
             fontsize=15, fontweight='bold', color=GREEN, y=1.03)

plt.tight_layout()
plt.savefig('gk_long_ball_king.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n👑 LONG BALL KING: {king['player_name']}  —  {king['long_per_match']:.1f}/match  |  {king['long_accuracy']:.0f}% accuracy")

## 13 · Short Ball King — GK Short Pass Distribution

In [ ]:
# ── Short Ball King Chart ─────────────────────────────────────────────────────
short_passes = gk_passes[gk_passes['is_short']].copy()
short_passes['is_successful'] = short_passes['pass_outcome'].isin(['', 'Complete'])

short_stats = short_passes.groupby('player_name').agg(
    short_balls       = ('is_short',       'sum'),
    successful_short  = ('is_successful',  'sum'),
    avg_short_length  = ('pass_length',    'mean'),
).reset_index()
short_stats = short_stats.merge(gk_mp, left_on='player_name', right_on='gk_name', how='left').drop(columns='gk_name')
short_stats = short_stats[short_stats['matches_played'] >= MIN_MATCHES].copy()
short_stats['short_per_match'] = short_stats['short_balls'] / short_stats['matches_played'].replace(0, 1)
short_stats['short_accuracy']  = (short_stats['successful_short'] / short_stats['short_balls'].replace(0, 1) * 100)
short_stats = short_stats.sort_values('short_per_match', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Left: Short Passes per Match (horizontal bar)
colors_short = [GREEN if v >= short_stats['short_per_match'].median() else '#2d6a4f' for v in short_stats['short_per_match']]
bars = axes[0].barh(short_stats['player_name'], short_stats['short_per_match'],
                     color=colors_short, edgecolor='none', height=0.65)
for bar, val in zip(bars, short_stats['short_per_match']):
    axes[0].text(val + 0.1, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}', va='center', ha='left', fontsize=9, fontweight='bold', color=WHITE)
axes[0].set_xlabel('Short Passes per Match (<32m)', fontsize=11)
axes[0].set_title('Short Pass Frequency', fontsize=13, fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)
med_short = short_stats['short_per_match'].median()
axes[0].axvline(med_short, color=YELLOW, ls='--', lw=1.3, alpha=0.8, label=f'Median: {med_short:.1f}')
axes[0].legend(framealpha=0.25, fontsize=9)

# Right: Accuracy vs Volume scatter
sc = axes[1].scatter(
    short_stats['short_per_match'], short_stats['short_accuracy'],
    c=short_stats['avg_short_length'], cmap='YlGn',
    s=short_stats['short_balls'] * 2 + 50,
    edgecolors='white', linewidths=0.6, alpha=0.9
)
for _, row in short_stats.iterrows():
    axes[1].annotate(row['player_name'].split()[-1],
                     (row['short_per_match'], row['short_accuracy']),
                     xytext=(5, 4), textcoords='offset points',
                     fontsize=8, color=WHITE, alpha=0.9)
cbar = plt.colorbar(sc, ax=axes[1])
cbar.set_label('Avg Short Pass Length (m)', color=WHITE, fontsize=10)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=WHITE)
axes[1].set_xlabel('Short Passes per Match', fontsize=11)
axes[1].set_ylabel('Short Pass Accuracy (%)', fontsize=11)
axes[1].set_title('Accuracy vs Volume', fontsize=13, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

# Crown the king
s_king = short_stats.iloc[-1]
plt.suptitle(f'Euro 2024 — Short Ball King: {s_king["player_name"]}\n'
             f'{s_king["short_per_match"]:.1f} short passes/match  ·  {s_king["short_accuracy"]:.0f}% accuracy  ·  Avg {s_king["avg_short_length"]:.0f}m',
             fontsize=15, fontweight='bold', color=GREEN, y=1.03)

plt.tight_layout()
plt.savefig('gk_short_ball_king.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n👑 SHORT BALL KING: {s_king['player_name']}  —  {s_king['short_per_match']:.1f}/match  |  {s_king['short_accuracy']:.0f}% accuracy")

## 14 · Heat Maps — GK Pass Distribution & Shot Zones

In [ ]:
# ── Heat Map 1: Pass destination zones for Top 4 GKs ─────────────────────────
from mplsoccer import Pitch

top4_gks = gk_r.head(4)['gk_name'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
fig.patch.set_facecolor(BG)

pitch_hm = Pitch(pitch_type='statsbomb', pitch_color=PITCH, line_color='#c7d5cc', linewidth=1)

for idx, (ax, gk_name) in enumerate(zip(axes.flat, top4_gks)):
    gk_pass_data = gk_passes[gk_passes['player_name'] == gk_name].copy()
    end_x = pd.to_numeric(gk_pass_data['pass_end_x'], errors='coerce').dropna()
    end_y = pd.to_numeric(gk_pass_data['pass_end_y'], errors='coerce').dropna()

    common_idx = end_x.index.intersection(end_y.index)
    end_x = end_x.loc[common_idx]
    end_y = end_y.loc[common_idx]

    pitch_hm.draw(ax=ax)
    ax.set_facecolor(PITCH)

    if len(end_x) > 0:
        bin_stat = pitch_hm.bin_statistic(end_x.values, end_y.values, statistic='count', bins=(12, 8))
        pitch_hm.heatmap(bin_stat, ax=ax, cmap='hot', edgecolors='#30363d', linewidth=0.3, alpha=0.75)
        pitch_hm.scatter(end_x.values, end_y.values, ax=ax, s=8, color='white', alpha=0.15, zorder=2)

    n_long  = gk_pass_data['is_long'].sum()
    n_short = gk_pass_data['is_short'].sum()
    mp_val  = gk_mp[gk_mp['gk_name'] == gk_name]['matches_played'].values
    mp_str  = f"{int(mp_val[0])} MP" if len(mp_val) > 0 else ""

    ax.set_title(f'{gk_name}\n{len(gk_pass_data)} passes  ·  {n_long} long  ·  {n_short} short  ·  {mp_str}',
                 fontsize=11, fontweight='bold', color=WHITE, pad=8)

plt.suptitle('Euro 2024 — GK Pass Destination Heat Maps (Top 4 Ranked)',
             fontsize=16, fontweight='bold', color=WHITE, y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('gk_pass_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Heat Map 2: Shots Faced Zone Heat Map for #1 GK ──────────────────────────
top_gk = gk_r.iloc[0]['gk_name']
gk_shots_hm = shots[shots['gk_name'] == top_gk].copy()
gk_shots_hm['shot_x'] = pd.to_numeric(gk_shots_hm['shot_x'], errors='coerce')
gk_shots_hm['shot_y'] = pd.to_numeric(gk_shots_hm['shot_y'], errors='coerce')
gk_shots_hm = gk_shots_hm.dropna(subset=['shot_x', 'shot_y'])

pitch_shot = VerticalPitch(pitch_type='statsbomb', pitch_color='grass',
                           line_color='white', half=True, goal_type='box', linewidth=1.5)
fig, ax = pitch_shot.draw(figsize=(10, 12))
fig.set_facecolor('#1a472a')

if len(gk_shots_hm) > 0:
    bin_stat = pitch_shot.bin_statistic(
        gk_shots_hm['shot_x'].values, gk_shots_hm['shot_y'].values,
        statistic='count', bins=(6, 5)
    )
    pitch_shot.heatmap(bin_stat, ax=ax, cmap='YlOrRd', edgecolors='white',
                       linewidth=0.5, alpha=0.7)
    pitch_shot.label_heatmap(bin_stat, ax=ax, color=WHITE, fontsize=14,
                             fontweight='bold', str_format='{:.0f}')

    # Overlay scatter with outcome colors
    for outcome, color in [('Saved', '#00e676'), ('Goal', '#ff1744'), ('Blocked', '#ffea00')]:
        mask = gk_shots_hm['outcome'] == outcome
        if mask.sum() > 0:
            pitch_shot.scatter(
                gk_shots_hm.loc[mask, 'shot_x'],
                gk_shots_hm.loc[mask, 'shot_y'],
                s=gk_shots_hm.loc[mask, 'xg'] * 500 + 30,
                c=color, alpha=0.7, edgecolors='white', linewidths=0.5,
                label=f'{outcome} ({mask.sum()})', ax=ax, zorder=3
            )

total = len(gk_shots_hm)
ga = (gk_shots_hm['outcome'] == 'Goal').sum()
xg_tot = gk_shots_hm['xg'].sum()
prev = xg_tot - ga

fig.text(0.5, 0.97, f'Shot Danger Zone Heat Map — {top_gk}',
         ha='center', fontsize=16, fontweight='bold', color='white',
         path_effects=[pe.withStroke(linewidth=3, foreground='#1a472a')])
fig.text(0.5, 0.935,
         f'Shots: {total}  ·  Goals: {ga}  ·  xG: {xg_tot:.1f}  ·  Prevented: {prev:+.1f}',
         ha='center', fontsize=10, color='#c8e6c9',
         path_effects=[pe.withStroke(linewidth=2, foreground='#1a472a')])
fig.text(0.5, 0.02, 'Zone numbers = shot count  ·  Bubble size ∝ xG  ·  StatsBomb Open Data',
         ha='center', fontsize=8, color='#a5d6a7',
         path_effects=[pe.withStroke(linewidth=1.5, foreground='#1a472a')])

ax.legend(loc='upper left', fontsize=9, framealpha=0.6,
          facecolor='#1b5e20', edgecolor='white', labelcolor='white')

plt.tight_layout(rect=[0, 0.03, 1, 0.92])
plt.savefig(f'gk_shot_heatmap_{top_gk.replace(" ", "_")}.png', dpi=150,
            bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f"Saved: gk_shot_heatmap_{top_gk.replace(' ', '_')}.png")